# Fusing Overlap Operations

When you chain spatial operations like erode then dilate, each one adds a blockwise layer to the dask graph. `fused_overlap` runs them in a single `map_overlap` call, and `multi_overlap` does the same for kernels that produce multiple output bands.

In [ ]:
import numpy as np
import dask.array as da
import xarray as xr
import xrspatial
from xrspatial.utils import fused_overlap, multi_overlap

## fused_overlap: chained operations in one pass

Define two stage functions. Each takes a padded chunk and returns the unpadded interior.

In [ ]:
def smooth_interior(chunk):
    """3x3 mean filter. Takes (H+2, W+2), returns (H, W)."""
    from numpy.lib.stride_tricks import sliding_window_view
    windows = sliding_window_view(chunk, (3, 3))
    return np.nanmean(windows, axis=(-2, -1))

def threshold_interior(chunk):
    """Binary threshold. Takes (H+2, W+2), returns (H, W)."""
    interior = chunk[1:-1, 1:-1]
    return (interior > 0.5).astype(np.float32)

np.random.seed(42)
raw = np.random.rand(512, 512).astype(np.float32)
dem = xr.DataArray(da.from_array(raw, chunks=128), dims=['y', 'x'])

In [ ]:
# Fused: one map_overlap call
fused = fused_overlap(dem, (smooth_interior, 1), (threshold_interior, 1))

# Sequential: two map_overlap calls
step1 = dem.data.map_overlap(smooth_interior, depth=1, boundary=np.nan, trim=False, meta=np.array(()))
sequential = step1.map_overlap(threshold_interior, depth=1, boundary=np.nan, trim=False, meta=np.array(()))

print(f'Fused graph:      {len(dict(fused.data.__dask_graph__())):,} tasks')
print(f'Sequential graph: {len(dict(sequential.__dask_graph__())):,} tasks')

## multi_overlap: N outputs in one pass

In [ ]:
def gradient_kernel(chunk):
    """Compute dx and dy gradients. Takes (H+2, W+2), returns (2, H, W)."""
    dx = (chunk[1:-1, 2:] - chunk[1:-1, :-2]) / 2.0
    dy = (chunk[2:, 1:-1] - chunk[:-2, 1:-1]) / 2.0
    return np.stack([dx, dy], axis=0)

result = multi_overlap(dem, gradient_kernel, n_outputs=2, depth=1)
print(f'Output shape: {result.shape}')
print(f'Dimensions:   {result.dims}')
print(f'Graph tasks:  {len(dict(result.data.__dask_graph__())):,}')

In [ ]:
# Accessor syntax works too
fused_acc = dem.xrs.fused_overlap((smooth_interior, 1), (threshold_interior, 1))
multi_acc = dem.xrs.multi_overlap(gradient_kernel, n_outputs=2, depth=1)
print('Accessor: OK')